### Gold Layer

In [0]:
from pyspark.sql import functions as F

In [0]:
silver_df = spark.table("dev.taxi_db.silver_yellow_taxi")

In [0]:
gold_daily_kpi = (silver_df.groupBy(F.day('pickup_datetime').alias('day'))
                  .agg(
                      F.count('*').alias("total_trips"),
                      F.sum("total_amount").alias('total_revenue'),
                      F.round(F.avg('fare_amount'),2).alias("avg_fare"),
                      F.median('passenger_count').alias("avg_passenger"),
                      F.round(F.avg('trip_distance_miles'),2).alias("avg_distance")
                  ).orderBy("day")

)
gold_daily_kpi.display()

In [0]:
passenger_distribution = silver_df.groupBy('passenger_count','').agg(F.count('*'))

In [0]:
gold_hourly = silver_df.withColumn(
    "pickup_hour", F.hour("pickup_datetime")
).groupBy("pickup_hour").agg(
    F.count("*").alias("trip_count"),
    F.sum("total_amount").alias("revenue")
).orderBy('pickup_hour')

gold_hourly.display()

In [0]:
# most pickup location

gold_location = silver_df.groupBy("pickup_location_id").agg(
    F.count("*").alias("pickup_count"),
    F.sum("total_amount").alias("revenue")
).orderBy(F.desc('revenue'))
gold_location.display()

In [0]:
# 10 most popular routes
gold_routes = silver_df.groupBy(
    "pickup_location_id", "dropoff_location_id"
).agg(
    F.count("*").alias("trip_count"),
    F.avg("fare_amount").alias("avg_fare")
).orderBy(F.desc('avg_fare')).limit(10)

gold_routes.display()

In [0]:
# tip percentage

gold_tip = silver_df.withColumn(
    "tip_pct",
    F.try_divide(F.col("tip_amount"), F.col("fare_amount")) * 100
).groupBy("payment_type_id").agg(
    F.avg("tip_pct").alias("avg_tip_pct")
)
gold_tip.display()

In [0]:
gold_daily_kpi.write.format("delta") \
    .mode("overwrite") \
    .saveAsTable("dev.taxi_db.gold_daily_kpi")